# 🌱 Green-Code Optimizer — GRPO Training (Colab)

**OpenEnv India Hackathon 2026** — reproduces the training run for the [Green-Code Optimizer](https://huggingface.co/spaces/s123hree/green-code-optimizer-a100) environment.

**The pitch:** train an RL agent to refactor Python for *energy efficiency* — minimising CPU cycles and peak memory while preserving program logic. The reward is grounded in real runtime measurements + graphlet analysis of control flow, and CPU-time savings are converted into CO₂-savings on a live dashboard.

**Reward formula:**
```
R = S_test × (0.70·green_score + 0.30·compliance_score) − P_efficiency
```

**Runtime requirements:** an A100 (or any GPU with `compute_capability >= 8.0`).  
On Colab pick `Runtime → Change runtime type → GPU → A100` (or T4 if you must, but expect a much slower run).

## What this notebook does
1. Clones the repo from the Hugging Face Space.
2. Installs Unsloth + TRL + OpenEnv.
3. Runs `training/train_grpo.py` (Qwen2.5-Coder-7B + QLoRA + GRPO).
4. Saves loss / reward plots to `assets/training_curves.png`.
5. Pushes the trained adapter to the Hugging Face Hub.

## 1. Check GPU

In [ ]:
!nvidia-smi

## 2. Clone the repo

In [ ]:
!git clone https://github.com/bcde123/Meta-Round2.git /content/green-code-optimizer
%cd /content/green-code-optimizer

## 3. Install dependencies

Unsloth provides pinned wheels that play nicely with Colab's CUDA.

In [ ]:
import os, sys, glob, shutil
# Clean up any corrupted PIP torch installs (~orch) caused by interrupted installations
for sp in [p for p in sys.path if 'site-packages' in p]:
    for broken in glob.glob(os.path.join(sp, "~orch*")) + glob.glob(os.path.join(sp, "~orchaudio*")) + glob.glob(os.path.join(sp, "~orchvision*")):
        shutil.rmtree(broken, ignore_errors=True)

!pip install -q --upgrade pip
!pip install --upgrade --force-reinstall torch==2.6.0
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps trl peft accelerate bitsandbytes
!pip install "numpy<2.0.0"
!pip uninstall -y vllm
!pip install -q -r requirements.txt

## 4. Authenticate with the HF Hub

Used to upload the trained adapter at the end of the run. Generate a **write** token at https://huggingface.co/settings/tokens.

In [ ]:
import os
from getpass import getpass
os.environ["HF_TOKEN"] = getpass("HF write token: ")
os.environ["HF_ADAPTER_REPO"] = "s123hree/green-code-optimizer-adapter-7b"
os.environ["WANDB_DISABLED"] = "true"

## 5. Run training

On A100 this takes **~1-2 hours**. The script:
- Loads `Qwen/Qwen2.5-Coder-7B-Instruct` in 4-bit
- Attaches LoRA adapters (`r=8`)
- Runs **GRPO for 200 steps** with 4 generations / step
- Saves `assets/training_curves.png` (loss + reward)
- Uploads the adapter to `$HF_ADAPTER_REPO`

In [ ]:
!python training/train_grpo.py

## 6. Inspect the training curves

In [ ]:
from IPython.display import Image
Image("assets/training_curves.png")

## 7. Quick inference smoke-test

In [ ]:
import requests, subprocess, time, signal

server = subprocess.Popen(["uvicorn", "server:app", "--host", "0.0.0.0", "--port", "7860"])
try:
    time.sleep(15)
    print(requests.get("http://localhost:7860/health").json())
    ep = requests.post("http://localhost:7860/reset").json()
    print("Episode:", ep.get("episode_id"), "steps_remaining:", ep.get("steps_remaining"))
finally:
    server.send_signal(signal.SIGINT)
    server.wait()